# Valuing Actions by Estimating Probabilities (VAEP)

## Import Libraries

In [1]:
# Ensure that no .pyc files are generated
import sys

sys.dont_write_bytecode = True

In [ ]:
import pandas as pd
import socceraction.vaep.features as fs
import xgboost as xgb
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from tqdm import tqdm

from config import paths

## Load SPADL Data

In [3]:
all_games_df = pd.read_hdf(paths.SPADL_H5, "games")

## Split Data for Training and Testing

In [4]:
# For lack of data, we will use the same games for training and testing
train_games_df = all_games_df
test_games_df = all_games_df

In [ ]:
def get_XY(games_df) -> tuple[pd.DataFrame, pd.DataFrame]:
    # Get the features for all games
    x_fns = [
        fs.actiontype,
        fs.actiontype_onehot,
        # fs.bodypart,
        fs.bodypart_onehot,
        fs.result,
        fs.result_onehot,
        fs.goalscore,
        fs.startlocation,
        fs.endlocation,
        fs.movement,
        fs.space_delta,
        fs.startpolar,
        fs.endpolar,
        fs.team,
        # fs.time,
        fs.time_delta,
        # fs.actiontype_result_onehot,
    ]
    x_cols = fs.feature_column_names(x_fns, nb_prev_actions=1)
    X = []

    for game_id in tqdm(games_df.game_id, desc="Selecting features"):
        x_i = pd.read_hdf(paths.FEATURES_H5, f"game_{game_id}")
        X.append(x_i[x_cols])

    X = pd.concat(X).reset_index(drop=True)

    # Get the labels for all games
    y_cols = ["scores", "concedes"]
    Y = []

    for game_id in tqdm(games_df.game_id, desc="Selecting labels"):
        y_i = pd.read_hdf(paths.LABELS_H5, f"game_{game_id}")
        Y.append(y_i[y_cols])

    Y = pd.concat(Y).reset_index(drop=True)

    return X, Y

In [ ]:
X, Y = get_XY(train_games_df)

Selecting labels: 100%|██████████| 2085/2085 [00:18<00:00, 110.30it/s]


## Train XGBoost Models for Scoring and Conciding

In [8]:
models = {}

for col in list(Y.columns):
    model = xgb.XGBClassifier(n_estimators=50, max_depth=3, n_jobs=-3, verbosity=1, enable_categorical=True)
    model.fit(X, Y[col])
    models[col] = model

## Evaluate the Models

In [9]:
test_x = X
test_y = Y
y_hat = pd.DataFrame()


def evaluate(y, y_hat):
    p = sum(y) / len(y)
    base = [p] * len(y)
    brier = brier_score_loss(y, y_hat)
    print(f"  Brier score: {brier:.5f} ({brier / brier_score_loss(y, base):.5f})")
    ll = log_loss(y, y_hat)
    print(f"  Log Loss score: {ll:.5f} ({ll / log_loss(y, base):.5f})")
    print(f"  ROC AUC: {roc_auc_score(y, y_hat):.5f}")


for col in test_y.columns:
    y_hat[col] = [p[1] for p in models[col].predict_proba(test_x)]
    print(f"### Y: {col} ###")
    evaluate(test_y[col], y_hat[col])

### Y: scores ###
  Brier score: 0.00894 (0.84809)
  Log Loss score: 0.04620 (0.78319)
  ROC AUC: 0.81862
### Y: concedes ###
  Brier score: 0.00232 (1.04231)
  Log Loss score: 0.01762 (1.11154)
  ROC AUC: 0.73014


## Save predictions to HDF5 file

In [10]:
# Get the game_id for each action in the all_games_df
A = []

for game_id in tqdm(all_games_df.game_id, "Loading game ids"):
    a_i = pd.read_hdf(paths.SPADL_H5, f"actions/game_{game_id}")
    A.append(a_i[["game_id"]])

A = pd.concat(A).reset_index(drop=True)

Loading game ids: 100%|██████████| 2085/2085 [00:38<00:00, 53.98it/s]


In [ ]:
# Group the predictions by game_id and save them to an HDF5 file
grouped_predictions = pd.concat([A, y_hat], axis=1).groupby("game_id")  # type: ignore

with pd.HDFStore(paths.PREDICTIONS_H5) as prediction_store:
    for game_id, df in tqdm(grouped_predictions, desc="Saving predictions per game"):
        df = df.reset_index(drop=True)
        prediction_store.put(f"game_{int(game_id)}", df[y_hat.columns])

Saving predictions per game: 100%|██████████| 2085/2085 [00:12<00:00, 163.94it/s]
